# Google Colab: T4 GPU setup

1. **Runtime → Change runtime type**
   - **Hardware accelerator**: `GPU`
   - On **Colab Pro/Pro+**, you can pick **T4** explicitly when offered; the free tier usually assigns a **T4** when you choose GPU, but the exact GPU is not guaranteed.
2. Run the cells **top to bottom** once per new session.

This notebook checks the driver, confirms PyTorch sees the GPU, installs a **CUDA-enabled JAX** stack (needed for this repo’s `jax` / `jaxlib` pins), and optionally installs `requirements.txt` if you upload or clone the project.

In [ ]:
# Driver and GPU model (expect NVIDIA T4 on many GPU runtimes)
!nvidia-smi

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])

import torch

# Colab usually ships CUDA PyTorch; only reinstall if the runtime has no GPU build.
if not torch.cuda.is_available():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "torch",
            "torchvision",
            "torchaudio",
            "--index-url",
            "https://download.pytorch.org/whl/cu124",
        ]
    )
    raise RuntimeError(
        "Installed CUDA PyTorch. Use Runtime → Restart session, then run all cells again."
    )

assert torch.cuda.is_available(), "No CUDA GPU visible. Set Runtime → GPU and re-run."
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f"PyTorch {torch.__version__} | GPU: {name} | capability {cap}")
x = torch.randn(4096, 4096, device="cuda")
y = x @ x
print("GEMM on GPU:", float(y[0, 0].cpu()))

In [ ]:
# JAX with CUDA (matches typical Colab CUDA 12.x). Removes CPU-only jax if present.
import subprocess, sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "uninstall", "-y", "jax", "jaxlib"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "jax[cuda12]",
        "jaxlib",
        "-f",
        "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    ]
)

import jax
print("JAX", jax.__version__, "devices:", jax.devices())
x = jax.numpy.ones((4096, 4096))
y = x @ x
y.block_until_ready()
print("JAX matmul ok on", y.device())

## Optional: project dependencies (`graph-rerank-pipeline`)

- **Clone** your repo (replace URL) *or* **Upload** `requirements.txt` to the Colab file browser, then adjust the path in the next cell.
- For **API keys** (`cohere`, etc.), use **Secrets** (key icon) or `os.environ` — do not commit keys.

In [ ]:
# Uncomment one approach:

# !git clone https://github.com/YOUR_ORG/graph-rerank-pipeline.git
# %cd graph-rerank-pipeline
# !pip install -q -r requirements.txt

# Or, if you uploaded requirements.txt to /content:
# !pip install -q -r /content/requirements.txt

print("Edit this cell to clone/upload, then install requirements.")